# Diffusion Models for Fast Calorimeter Shower Simulation - Set Transformer Denoiser

## Overview

This notebook is a **direct continuation of notebook 03**.
We keep the same dataset, preprocessing, DDPM formulation, and evaluation workflow, and only change the denoiser from a per-hit MLP to a **Set Transformer** with masked self-attention.

## Learning Objectives (Delta vs notebook 03)

By the end of this notebook you will be able to:

1. Explain why cross-hit attention is useful for calorimeter point clouds.
2. Implement a Set Transformer denoiser conditioned on $p_{z0}$.
3. Compare transformer and MLP denoisers under the same training setup.
4. Quantify gains using the same HEP metrics introduced in notebook 03.

## Experiment Context

Same LUXE edm4hep source and feature set as notebook 03 (`x`, `y`, `z`, `edep`, with conditioning on `pz0`).

## DDPM Recap (See notebook 03 for full derivation)

The forward/reverse DDPM equations, noise schedule, and masked noise-prediction objective are unchanged from notebook 03.

In notebook 04, the **only modeling change** is the denoiser architecture:
- notebook 03: per-hit MLP (independent hit processing)
- notebook 04: Set Transformer (cross-hit attention with padding mask)

Use notebook 03 for the full mathematical background; here we focus on what attention changes in practice.

In [ ]:
# Core imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ROOT I/O
import uproot

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Plot styling
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 5)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Configure edm4hep Input (Same as notebook 03)

The ROOT branch mapping and edm4hep parsing are identical to notebook 03.
Keep this section as a reproducibility check, but no new concepts are introduced here.

In [ ]:
# Get the data only if missing locally
files_to_fetch = [
    ("luxe_positron_gun.edm4hep.root", "https://www.desy.de/~fmeloni/resources/luxe_positron_gun.edm4hep.root"),
    ("luxe_positron_6GeV.edm4hep.root", "https://www.desy.de/~fmeloni/resources/luxe_positron_6GeV.edm4hep.root"),
]

for local_name, url in files_to_fetch:
    if os.path.exists(local_name):
        print(f"Found local file: {local_name}")
    else:
        print(f"Downloading {local_name}...")
        os.system(f'wget -O "{local_name}" "{url}"')


In [ ]:
# --- User configuration ---
ROOT_PATH = "luxe_positron_gun.edm4hep.root"
#ROOT_PATH = "luxe_positron_6GeV.edm4hep.root"
TREE_NAME = "events"

BRANCH_MAP = {
    "x": "PixelSiEcalCollection/PixelSiEcalCollection.position.x",
    "y": "PixelSiEcalCollection/PixelSiEcalCollection.position.y",
    "z": "PixelSiEcalCollection/PixelSiEcalCollection.position.z",
    "edep": "PixelSiEcalCollection/PixelSiEcalCollection.energy",
    "mc_pz": "MCParticles/MCParticles.momentum.z",
}

print("Configured ROOT file:", ROOT_PATH)
print("Configured tree:", TREE_NAME)
print("Branch mapping:")
for k, v in BRANCH_MAP.items():
    print(f"  {k:>6} -> {v}")

In [ ]:
# Optional helper: inspect available trees and branches
if os.path.exists(ROOT_PATH):
    with uproot.open(ROOT_PATH) as f:
        print("Top-level keys:")
        print(f.keys())

        if TREE_NAME in f:
            tree = f[TREE_NAME]
            print(f"\nBranches in '{TREE_NAME}':")
            print(tree.keys())
        else:
            print(f"\nTree '{TREE_NAME}' not found. Choose one from keys above.")
else:
    print(f"File not found: {ROOT_PATH}")
    print("Update ROOT_PATH and rerun.")

## Load Hit-Level Data (Same as notebook 03)

Data flattening from jagged arrays and construction of the hit-level DataFrame are unchanged.
We reuse the same columns and conditional variable definition (`mc_pz0`).

In [ ]:
required_keys = ["x", "y", "z", "edep", "mc_pz"]
missing = [k for k in required_keys if k not in BRANCH_MAP]
if missing:
    raise ValueError(f"Missing branch mappings for: {missing}")

if not os.path.exists(ROOT_PATH):
    raise FileNotFoundError(f"ROOT file does not exist: {ROOT_PATH}")

with uproot.open(ROOT_PATH) as f:
    if TREE_NAME not in f:
        raise KeyError(f"Tree '{TREE_NAME}' not found in file. Available keys: {f.keys()}")
    tree = f[TREE_NAME]

    branch_names = [BRANCH_MAP[k] for k in required_keys]
    arrays = tree.arrays(branch_names, library="np")

# Flatten per-event jagged arrays into a hit-level DataFrame
records = []
n_events = len(arrays[BRANCH_MAP["x"]])

for event_idx in range(n_events):
    x_evt = np.asarray(arrays[BRANCH_MAP["x"]][event_idx], dtype=np.float32)
    y_evt = np.asarray(arrays[BRANCH_MAP["y"]][event_idx], dtype=np.float32)
    z_evt = np.asarray(arrays[BRANCH_MAP["z"]][event_idx], dtype=np.float32)
    e_evt = np.asarray(arrays[BRANCH_MAP["edep"]][event_idx], dtype=np.float32)
    pz_evt = np.asarray(arrays[BRANCH_MAP["mc_pz"]][event_idx], dtype=np.float32)

    n_hits = min(len(x_evt), len(y_evt), len(z_evt), len(e_evt))
    if n_hits == 0:
        continue

    # First MC particle pz in the event; skip event if no MC particles
    if len(pz_evt) == 0:
        continue

    mc_pz0 = float(pz_evt[0])

    event_df = pd.DataFrame({
        "event_id": np.full(n_hits, event_idx, dtype=np.int32),
        "x": x_evt[:n_hits],
        "y": y_evt[:n_hits],
        "z": z_evt[:n_hits],
        "edep": e_evt[:n_hits],
        "mc_pz0": np.full(n_hits, mc_pz0, dtype=np.float32),
    })
    records.append(event_df)

if not records:
    raise ValueError("No hits were loaded from the configured branches.")

hits_df = pd.concat(records, ignore_index=True)
print("Loaded hits:", len(hits_df))
print("Loaded events with at least one hit:", hits_df["event_id"].nunique())
print(hits_df.head(5))

In [ ]:
# Basic cleaning
hits_df = hits_df.replace([np.inf, -np.inf], np.nan).dropna().copy()
hits_df = hits_df[hits_df["edep"] > 0].copy()

# Use log10(edep) as the training feature while keeping linear edep for diagnostics
EDep_EPS = 1e-12
hits_df["edep_log10"] = np.log10(np.clip(hits_df["edep"].to_numpy(dtype=np.float32), EDep_EPS, None))

print("After cleaning:", hits_df.shape)
print(hits_df[["x", "y", "z", "edep", "edep_log10", "mc_pz0"]].describe().T[["mean", "std", "min", "max"]])

## Build Training Tensors (Same representation as notebook 03)

Point-cloud tensorization, padding/masking, feature transforms, and normalization are unchanged.
Keeping the exact same tensor pipeline ensures a fair MLP vs Set Transformer comparison.

In [ ]:
FEATURE_COLUMNS = ["x", "y", "z", "edep_log10"]

# Limit sequence length for memory efficiency in an initial draft
MAX_HITS = 160
MIN_HITS_PER_EVENT = 60

def build_event_tensors(df, max_hits=256, min_hits=10):
    X_list = []
    M_list = []
    E_list = []

    grouped = df.groupby("event_id", sort=False)
    for event_id, g in grouped:
        if len(g) < min_hits:
            continue

        # Shuffle hits in each event to avoid positional bias
        g = g.sample(frac=1.0, random_state=SEED)

        feats = g[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        feats = feats[:max_hits]

        n = feats.shape[0]
        mask = np.zeros((max_hits, 1), dtype=np.float32)
        mask[:n, 0] = 1.0

        padded = np.zeros((max_hits, feats.shape[1]), dtype=np.float32)
        padded[:n] = feats

        # Conditioning variable: first MC particle pz
        cond_vec = np.array([g["mc_pz0"].iloc[0]], dtype=np.float32)

        X_list.append(padded)
        M_list.append(mask)
        E_list.append(cond_vec)

    if len(X_list) == 0:
        raise ValueError(
            "No events passed the current filters. "
            "Try reducing MIN_HITS_PER_EVENT or check input branch mapping."
        )

    X = np.stack(X_list)
    M = np.stack(M_list)
    E = np.stack(E_list)
    return X, M, E

X_raw, M_raw, E_raw = build_event_tensors(hits_df, max_hits=MAX_HITS, min_hits=MIN_HITS_PER_EVENT)
print("Using features:", FEATURE_COLUMNS)
print("Conditioning variable: [mc_pz0]")
print("X_raw shape:", X_raw.shape)
print("M_raw shape:", M_raw.shape)
print("E_raw shape:", E_raw.shape)

In [ ]:
# Train/validation split and feature normalization
n_events = X_raw.shape[0]
perm = np.random.permutation(n_events)
train_size = int(0.8 * n_events)

idx_train = perm[:train_size]
idx_val = perm[train_size:]

X_train_raw, X_val_raw = X_raw[idx_train], X_raw[idx_val]
M_train, M_val = M_raw[idx_train], M_raw[idx_val]
E_train_raw, E_val_raw = E_raw[idx_train], E_raw[idx_val]

# Compute normalization only on non-padded training hits
masked_train = X_train_raw[M_train[..., 0] > 0.5]
feat_mean = masked_train.mean(axis=0, keepdims=True)
feat_std = masked_train.std(axis=0, keepdims=True) + 1e-6

X_train = (X_train_raw - feat_mean) / feat_std
X_val = (X_val_raw - feat_mean) / feat_std

E_mean = E_train_raw.mean(axis=0, keepdims=True)
E_std = E_train_raw.std(axis=0, keepdims=True) + 1e-6
E_train = (E_train_raw - E_mean) / E_std
E_val = (E_val_raw - E_mean) / E_std

print("Train events:", len(X_train))
print("Validation events:", len(X_val))
print("Feature mean:", feat_mean)
print("Feature std:", feat_std)

## Define the Set Transformer Denoising Network and Training Procedure

### Architecture: Set Transformer Denoiser

The denoiser is a permutation-equivariant Set Transformer built from stacked `SetAttentionBlock` layers.
Each block applies masked multi-head self-attention across hits, then a position-wise feed-forward network with residual connections.

Pipeline for noisy point clouds $\mathbf{X}_t \in \mathbb{R}^{B \times H \times F}$:

1. Input projection: feature dimension $F \rightarrow D$.
2. Context injection: add time+conditioning embedding to each hit slot.
3. Repeated Set Attention Blocks with padding-aware attention masks.
4. Output projection back to $F$ to predict $\hat{\epsilon}$ per hit.

Key difference vs notebook 03: each hit prediction can depend on all other real hits in the event.

### Time Embedding and Loss

Same sinusoidal time embedding and masked noise-prediction MSE objective as notebook 03.
Only the denoiser architecture changes.

In [ ]:
# Diffusion hyperparameters
N_FEATURES = X_train.shape[-1]
HIDDEN_DIM = 128
TIMESTEPS = 300
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 5e-4
EARLY_STOP_PATIENCE = 5

# Linear beta schedule
betas = np.linspace(1e-4, 0.02, TIMESTEPS, dtype=np.float32)
alphas = 1.0 - betas
alpha_bars = np.cumprod(alphas)

betas_tf = tf.constant(betas, dtype=tf.float32)
alphas_tf = tf.constant(alphas, dtype=tf.float32)
alpha_bars_tf = tf.constant(alpha_bars, dtype=tf.float32)

def gather_t(values, t):
    # values: (T,), t: (B,)
    v = tf.gather(values, t)
    return tf.reshape(v, (-1, 1, 1))

def time_embedding(t, dim=64):
    half = dim // 2
    freqs = tf.exp(tf.linspace(tf.math.log(1.0), tf.math.log(1000.0), half))
    angles = tf.cast(tf.expand_dims(t, -1), tf.float32) / freqs
    return tf.concat([tf.sin(angles), tf.cos(angles)], axis=-1)

class SetAttentionBlock(layers.Layer):
    def __init__(self, hidden, num_heads=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=hidden // num_heads,
            dropout=dropout,
        )
        self.drop1 = layers.Dropout(dropout)

        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = keras.Sequential([
            layers.Dense(ff_mult * hidden, activation="swish"),
            layers.Dropout(dropout),
            layers.Dense(hidden),
        ])

    def call(self, h, attn_mask, training=False):
        h_norm = self.norm1(h)
        attn_out = self.attn(
            query=h_norm,
            value=h_norm,
            key=h_norm,
            attention_mask=attn_mask,
            training=training,
        )
        h = h + self.drop1(attn_out, training=training)
        h = h + self.ffn(self.norm2(h), training=training)
        return h

class NoisePredictor(keras.Model):
    def __init__(self, n_features, hidden=64, num_heads=8, num_blocks=3):
        super().__init__()
        self.time_dense = layers.Dense(hidden, activation="swish")
        self.cond_dense = layers.Dense(hidden, activation="swish")
        self.input_proj = layers.Dense(hidden)

        self.blocks = [
            SetAttentionBlock(hidden=hidden, num_heads=num_heads, ff_mult=2, dropout=0.1)
            for _ in range(num_blocks)
        ]

        self.out_norm = layers.LayerNormalization(epsilon=1e-6)
        self.noise_out = layers.Dense(n_features)

    def call(self, x_t, t, cond, hit_mask=None, training=False):
        # x_t: (B, H, F), t: (B,), cond: (B, C), hit_mask: (B, H)
        t_emb = time_embedding(t, dim=64)
        context = self.time_dense(t_emb) + self.cond_dense(cond)  # (B, D)

        h = self.input_proj(x_t) + context[:, None, :]

        if hit_mask is None:
            hit_mask = tf.ones(tf.shape(x_t)[:2], dtype=tf.float32)

        attn_mask = tf.cast(hit_mask[:, :, None] * hit_mask[:, None, :], tf.bool)

        for block in self.blocks:
            h = block(h, attn_mask=attn_mask, training=training)

        eps_pred = self.noise_out(self.out_norm(h))
        return eps_pred * hit_mask[:, :, None]

model = NoisePredictor(n_features=N_FEATURES, hidden=HIDDEN_DIM)
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

# Build tf.data datasets
train_ds = tf.data.Dataset.from_tensor_slices((
    X_train.astype(np.float32),
    M_train.astype(np.float32),
    E_train.astype(np.float32),
))
train_ds = train_ds.shuffle(len(X_train), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((
    X_val.astype(np.float32),
    M_val.astype(np.float32),
    E_val.astype(np.float32),
))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
@tf.function
def train_step(x0, mask, cond):
    bsz = tf.shape(x0)[0]
    t = tf.random.uniform((bsz,), minval=0, maxval=TIMESTEPS, dtype=tf.int32)

    eps = tf.random.normal(tf.shape(x0))
    a_bar_t = gather_t(alpha_bars_tf, t)

    x_t = tf.sqrt(a_bar_t) * x0 + tf.sqrt(1.0 - a_bar_t) * eps

    with tf.GradientTape() as tape:
        hit_mask = tf.reduce_max(mask, axis=-1)
        eps_pred = model(x_t, t, cond, hit_mask=hit_mask, training=True)
        sq_err = tf.square(eps - eps_pred)
        masked_sq_err = sq_err * mask
        loss = tf.reduce_sum(masked_sq_err) / (
            tf.reduce_sum(mask) * tf.cast(tf.shape(x0)[-1], tf.float32) + 1e-8
        )

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def val_step(x0, mask, cond):
    bsz = tf.shape(x0)[0]
    t = tf.random.uniform((bsz,), minval=0, maxval=TIMESTEPS, dtype=tf.int32)

    eps = tf.random.normal(tf.shape(x0))
    a_bar_t = gather_t(alpha_bars_tf, t)
    x_t = tf.sqrt(a_bar_t) * x0 + tf.sqrt(1.0 - a_bar_t) * eps

    hit_mask = tf.reduce_max(mask, axis=-1)
    eps_pred = model(x_t, t, cond, hit_mask=hit_mask, training=False)
    sq_err = tf.square(eps - eps_pred)
    masked_sq_err = sq_err * mask
    loss = tf.reduce_sum(masked_sq_err) / (
        tf.reduce_sum(mask) * tf.cast(tf.shape(x0)[-1], tf.float32) + 1e-8
    )
    return loss

train_losses, val_losses = [], []

# Early stopping state
best_val_loss = float("inf")
best_weights = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    epoch_train = []
    for xb, mb, cb in train_ds:
        loss = train_step(xb, mb, cb)
        epoch_train.append(loss.numpy())

    epoch_val = []
    for xb, mb, cb in val_ds:
        vloss = val_step(xb, mb, cb)
        epoch_val.append(vloss.numpy())

    tr = float(np.mean(epoch_train))
    va = float(np.mean(epoch_val))

    train_losses.append(tr)
    val_losses.append(va)

    if va < best_val_loss:
        best_val_loss = va
        best_weights = model.get_weights()
        patience_counter = 0
        improved_marker = " *"
    else:
        patience_counter += 1
        improved_marker = f" (no improvement {patience_counter}/{EARLY_STOP_PATIENCE})"

    print(
        f"Epoch {epoch:03d} | train={tr:.5f} val={va:.5f}"
        + improved_marker
    )

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}. Restoring best weights.")
        model.set_weights(best_weights)
        break

# Restore best weights if training finished without early stop
if best_weights is not None and patience_counter < EARLY_STOP_PATIENCE:
    model.set_weights(best_weights)

plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Diffusion training curves")
plt.show()


In [ ]:
# Reverse diffusion sampling

def sample_showers(n_samples, cond_pz_norm):
    """Generate showers using ancestral DDPM sampling."""
    x_t = tf.random.normal((n_samples, MAX_HITS, N_FEATURES))
    cond = tf.convert_to_tensor(cond_pz_norm.astype(np.float32))
    full_hit_mask = tf.ones((n_samples, MAX_HITS), dtype=tf.float32)

    for step in reversed(range(TIMESTEPS)):
        t = tf.fill((n_samples,), step)
        eps_pred = model(x_t, t, cond, hit_mask=full_hit_mask, training=False)

        alpha_t = alphas_tf[step]
        alpha_bar_t = alpha_bars_tf[step]
        beta_t = betas_tf[step]

        # DDPM mean update
        coef1 = 1.0 / tf.sqrt(alpha_t)
        coef2 = beta_t / tf.sqrt(1.0 - alpha_bar_t)
        mean = coef1 * (x_t - coef2 * eps_pred)

        if step > 0:
            z = tf.random.normal(tf.shape(x_t))
            sigma = tf.sqrt(beta_t)
            x_t = mean + sigma * z
        else:
            x_t = mean

    return x_t.numpy()

# Example: condition on median training pz0 (normalized space, same shape as training cond)
median_cond = np.median(E_train, axis=0, keepdims=True)
cond_med = np.repeat(median_cond, 16, axis=0).astype(np.float32)

samples_norm = sample_showers(16, cond_med)

# De-normalize using training feature stats
samples_denorm = samples_norm * feat_std + feat_mean

In [ ]:

# Compare one real shower and one generated shower conditioned on the same pz0
real_event_id = int(hits_df["event_id"].iloc[0])
real = hits_df[hits_df["event_id"] == real_event_id]

# Get the conditioning value for this specific real event and normalise it
mc_pz0 = float(real["mc_pz0"].iloc[0])
cond_norm = np.array([[(mc_pz0 - float(E_mean[0, 0])) / float(E_std[0, 0])]], dtype=np.float32)

# Generate one shower conditioned on the same pz0 as the real event
sample_norm = sample_showers(n_samples=1, cond_pz_norm=cond_norm)[0]

# De-normalise generated features
sample = sample_norm * feat_std + feat_mean
samp_x = sample[:, FEATURE_COLUMNS.index("x")]
samp_y = sample[:, FEATURE_COLUMNS.index("y")]
samp_z = sample[:, FEATURE_COLUMNS.index("z")]
samp_edep_log10 = sample[:, FEATURE_COLUMNS.index("edep_log10")]
samp_edep = np.power(10.0, samp_edep_log10)

# Plot
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection="3d")
real_sizes = 10 + 120 * (real["edep"] / (real["edep"].max() + 1e-12))
sc1 = ax1.scatter(real["x"], real["z"], real["y"], c=real["edep"], s=real_sizes, cmap="viridis", alpha=0.8)
ax1.set_title(f"Real shower (event {real_event_id}, pz0={mc_pz0:.1f} MeV)")
ax1.set_xlabel("x")
ax1.set_ylabel("z")
ax1.set_zlabel("y")
plt.colorbar(sc1, ax=ax1, pad=0.1, label="edep")

ax2 = fig.add_subplot(122, projection="3d")
samp_sizes = 10 + 120 * (samp_edep / (samp_edep.max() + 1e-12))
sc2 = ax2.scatter(samp_x, samp_z, samp_y, c=samp_edep, s=samp_sizes, cmap="plasma", alpha=0.8)
plt.colorbar(sc2, ax=ax2, pad=0.1, label="generated edep")
ax2.set_title(f"Generated shower (diffusion, pz0={mc_pz0:.1f} MeV)")
ax2.set_xlabel("x")
ax2.set_ylabel("z")
ax2.set_zlabel("y")

plt.tight_layout()
plt.show()


### Exercises — Training Diagnostics and Sampling

After running the training and sampling cells, answer the following:

1. **Diffusion timesteps.** The schedule has `TIMESTEPS = 300`. Reduce it to `100` and retrain. How does the generated shower quality change? What is the trade-off between number of timesteps and sampling speed?
2. **Number of attention blocks.** Try `num_blocks = 1` and `num_blocks = 5`. How do the validation loss and shower profiles change? At what point does adding more blocks stop helping?
3. **Number of attention heads.** Change `num_heads` from 4 to 1 (single-head) and then to 8. Does multi-head attention help the model capture the hit-to-hit correlation structure?
4. **Attention masking.** Remove the `hit_mask` argument when calling `model(...)` (so all slots — including padding — attend to each other). Does this degrade shower quality? Why?
5. **Comparison with the MLP baseline.** Run the Wasserstein-distance table for both architectures. Which metric improves the most with cross-hit attention?


## Quantitative Validation (Metrics reused from notebook 03)

Use the same shower-quality metrics from notebook 03:
- energy response and resolution
- longitudinal/lateral profiles
- event-shape moments

The focus here is **relative performance** of Set Transformer vs MLP under identical metrics and binning.

In [ ]:
# --- Quantitative validation: real vs generated ---

def decode_generated_batch(
    gen_norm_batch,
    feat_mean,
    feat_std,
    feature_columns,
    event_id_offset=10_000_000,
):
    """Convert generated normalized tensors to a hit-level DataFrame."""
    gen = gen_norm_batch * feat_std + feat_mean
    ix = feature_columns.index("x")
    iy = feature_columns.index("y")
    iz = feature_columns.index("z")
    ie_log = feature_columns.index("edep_log10")

    rows = []
    for i in range(gen.shape[0]):
        x = gen[i, :, ix]
        y = gen[i, :, iy]
        z = gen[i, :, iz]
        e_log = gen[i, :, ie_log]
        e = np.power(10.0, e_log)

        df_i = pd.DataFrame({
            "event_id": np.full(len(e), event_id_offset + i, dtype=np.int32),
            "x": x,
            "y": y,
            "z": z,
            "edep": e,
        })
        rows.append(df_i)

    if not rows:
        return pd.DataFrame(columns=["event_id", "x", "y", "z", "edep"])

    return pd.concat(rows, ignore_index=True)


def shower_metrics(df):
    """Compute event-level shower moments and energy summaries."""
    out = []
    grouped = df.groupby("event_id", sort=False)
    for eid, g in grouped:
        e = g["edep"].to_numpy(np.float64)
        x = g["x"].to_numpy(np.float64)
        y = g["y"].to_numpy(np.float64)
        z = g["z"].to_numpy(np.float64)

        e_sum = e.sum()
        if e_sum <= 0:
            continue

        x_bar = np.sum(e * x) / e_sum
        y_bar = np.sum(e * y) / e_sum
        z_bar = np.sum(e * z) / e_sum

        r2 = (x - x_bar) ** 2 + (y - y_bar) ** 2
        lateral_rms = np.sqrt(np.sum(e * r2) / e_sum)
        long_rms = np.sqrt(np.sum(e * (z - z_bar) ** 2) / e_sum)

        out.append({
            "event_id": int(eid),
            "n_hits": int(np.count_nonzero(e > 0.0)),
            "total_edep": float(e_sum),
            "x_bar": float(x_bar),
            "y_bar": float(y_bar),
            "z_bar": float(z_bar),
            "lateral_rms": float(lateral_rms),
            "long_rms": float(long_rms),
        })

    return pd.DataFrame(out)


def profile_arrays(df):
    """Collect weighted arrays for longitudinal (z) and lateral (rho) profiles."""
    z_all, rho_all, e_all = [], [], []
    for _, g in df.groupby("event_id", sort=False):
        e = g["edep"].to_numpy(np.float64)
        if e.sum() <= 0:
            continue
        x = g["x"].to_numpy(np.float64)
        y = g["y"].to_numpy(np.float64)
        z = g["z"].to_numpy(np.float64)
        x_bar = np.sum(e * x) / e.sum()
        y_bar = np.sum(e * y) / e.sum()
        rho = np.sqrt((x - x_bar) ** 2 + (y - y_bar) ** 2)

        z_all.append(z)
        rho_all.append(rho)
        e_all.append(e)

    if not z_all:
        return np.array([]), np.array([]), np.array([])

    return np.concatenate(z_all), np.concatenate(rho_all), np.concatenate(e_all)


# Build real validation subset aligned with tensor split
valid_event_ids = [
    int(eid)
    for eid, g in hits_df.groupby("event_id", sort=False)
    if len(g) >= MIN_HITS_PER_EVENT
]
valid_event_ids = np.asarray(valid_event_ids, dtype=np.int32)

if len(idx_val) == 0:
    raise ValueError("Validation split is empty; cannot run quantitative validation.")

real_val_ids = valid_event_ids[idx_val]
real_val_df = hits_df[hits_df["event_id"].isin(real_val_ids)].copy()

# Sample generated showers using validation conditioning
n_eval = len(E_val)
gen_val_norm = sample_showers(n_samples=n_eval, cond_pz_norm=E_val)
gen_val_df = decode_generated_batch(gen_val_norm, feat_mean, feat_std, FEATURE_COLUMNS)

# Event-level metrics
real_metrics = shower_metrics(real_val_df)
gen_metrics = shower_metrics(gen_val_df)

if real_metrics.empty or gen_metrics.empty:
    raise ValueError("Metrics are empty. Check generated events and hit filtering.")

print("Real events used:", len(real_metrics))
print("Generated events used:", len(gen_metrics))
print("Mean n_hits (real):", float(real_metrics["n_hits"].mean()))
print("Mean n_hits (generated):", float(gen_metrics["n_hits"].mean()))

# Total energy response and resolution
real_mean_e = real_metrics["total_edep"].mean()
gen_mean_e = gen_metrics["total_edep"].mean()
response = gen_mean_e / (real_mean_e + 1e-12)
real_res = real_metrics["total_edep"].std(ddof=1) / (real_mean_e + 1e-12)
gen_res = gen_metrics["total_edep"].std(ddof=1) / (gen_mean_e + 1e-12)

print(f"Energy response (gen/real mean total edep): {response:.4f}")
print(f"Energy resolution real (sigma/mean): {real_res:.4f}")
print(f"Energy resolution gen  (sigma/mean): {gen_res:.4f}")

# Distribution comparisons
metrics_to_plot = ["n_hits", "total_edep", "lateral_rms", "long_rms"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, m in zip(axes.flatten(), metrics_to_plot):
    sns.histplot(real_metrics[m], bins=60, stat="density", element="step", fill=False, label="real", ax=ax)
    sns.histplot(gen_metrics[m], bins=60, stat="density", element="step", fill=False, label="generated", ax=ax)
    ax.set_title(m)
    ax.legend()
plt.tight_layout()
plt.show()

# Longitudinal and lateral profiles (energy-weighted, normalized)
z_real, rho_real, e_real = profile_arrays(real_val_df)
z_gen, rho_gen, e_gen = profile_arrays(gen_val_df)

z_min = min(z_real.min(), z_gen.min())
z_max = max(z_real.max(), z_gen.max())
r_max = max(rho_real.max(), rho_gen.max())

z_bins = np.linspace(z_min, z_max, 60)
r_bins = np.linspace(0.0, r_max + 1e-9, 60)

z_hist_real, _ = np.histogram(z_real, bins=z_bins, weights=e_real)
z_hist_gen, _ = np.histogram(z_gen, bins=z_bins, weights=e_gen)
r_hist_real, _ = np.histogram(rho_real, bins=r_bins, weights=e_real)
r_hist_gen, _ = np.histogram(rho_gen, bins=r_bins, weights=e_gen)

z_hist_real = z_hist_real / (z_hist_real.sum() + 1e-12)
z_hist_gen = z_hist_gen / (z_hist_gen.sum() + 1e-12)
r_hist_real = r_hist_real / (r_hist_real.sum() + 1e-12)
r_hist_gen = r_hist_gen / (r_hist_gen.sum() + 1e-12)

z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(z_centers, z_hist_real, label="real")
ax[0].plot(z_centers, z_hist_gen, label="generated")
ax[0].set_title("Longitudinal profile (z)")
ax[0].set_xlabel("z")
ax[0].set_ylabel("Normalized deposited energy")
ax[0].legend()

ax[1].plot(r_centers, r_hist_real, label="real")
ax[1].plot(r_centers, r_hist_gen, label="generated")
ax[1].set_title("Lateral profile (rho)")
ax[1].set_xlabel("rho")
ax[1].set_ylabel("Normalized deposited energy")
ax[1].legend()

plt.tight_layout()
plt.show()


### Quantitative Validation

As in notebook 03, compute distribution distances (for example Wasserstein-1) between real and generated samples.
In notebook 04, report these values side by side with the MLP baseline to isolate the impact of cross-hit attention.

In [ ]:
from scipy.stats import wasserstein_distance

metrics_to_compare = ["n_hits", "total_edep", "lateral_rms", "long_rms"]

# Collect per-architecture results.
# Add entries for additional architectures before re-running this cell.
arch_metrics = {
    "MLP denoiser": (real_metrics, gen_metrics),
    # "Set Transformer": (real_metrics_st, gen_metrics_st),  # uncomment once trained
}

wass_table = {}
for arch_name, (real_m, gen_m) in arch_metrics.items():
    wass_table[arch_name] = {}
    for metric in metrics_to_compare:
        r = real_m[metric].dropna().to_numpy(np.float64)
        g = gen_m[metric].dropna().to_numpy(np.float64)
        wass_table[arch_name][metric] = wasserstein_distance(r, g)

# Build a DataFrame and print a formatted table
wass_df = pd.DataFrame(wass_table, index=metrics_to_compare)
wass_df.index.name = "metric"

col_width = 18
header = f"{'metric':<{col_width}}" + "".join(
    f"{arch:>{col_width}}" for arch in wass_df.columns
)
sep = "-" * len(header)
print("\n(Wasserstein-1 distance; lower = better match to data)")
print(sep)
print(header)
print(sep)
for metric in metrics_to_compare:
    row = f"{metric:<{col_width}}"
    for arch in wass_df.columns:
        row += f"{wass_df.loc[metric, arch]:>{col_width}.4f}"
    print(row)
print(sep)


## Summary

Notebook 04 keeps the full notebook 03 pipeline fixed (data, DDPM objective, training protocol, and metrics) and changes only the denoiser:

1. Replace per-hit MLP with a masked Set Transformer.
2. Enable cross-hit interactions through multi-head self-attention.
3. Evaluate gains against the notebook 03 baseline with identical validation metrics.